<a href="https://colab.research.google.com/github/nothing248/notebooks/blob/main/notebooks/video_subtitle/remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!ls /kaggle/working

__notebook__.ipynb


In [2]:
# @title 查看cuda版本
!nvcc -V
!nvidia-smi

/bin/bash: line 1: nvcc: command not found
/bin/bash: line 1: nvidia-smi: command not found


In [3]:
import os
# @title 定义路径
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: 
    # 创建tmp
    os.makedirs('/kaggle/temp', exist_ok=True)
    # Kaggle 环境
    pro_path = '/kaggle/temp/video-subtitle-remover'
    exe_path = pro_path
    input_path = '/kaggle/input/datasets/xiaoyuetmyang/subtitle'
    output_path = '/kaggle/working'
elif "COLAB_RELEASE_TAG" in os.environ: 
    # colab 环境
    from google.colab import drive
    drive.mount('/content/drive')
    
    pro_path = '/content/drive/MyDrive/projects/video-subtitle-remover'
    exe_path = '/content/temp'
    input_path = f'{pro_path}/test'
    output_path = input_path

In [4]:
# @title 克隆仓库
!git clone --depth 1 https://github.com/YaoFANGUK/video-subtitle-remover {pro_path}

Cloning into '/kaggle/temp/video-subtitle-remover'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 230 (delta 14), reused 162 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 754.55 MiB | 36.97 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Updating files: 100% (184/184), done.


In [5]:
# @title 安装uv
%cd {exe_path}

import os

!pip install uv
os.environ["UV_VENV_CLEAR"] = "1"
!uv venv

/kaggle/temp/video-subtitle-remover
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/26.9 MB 60.9 MB/s eta 0:00:00
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [6]:
# @title 安装环境
!uv pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu118/ --python .venv/bin/python
!uv pip install torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu118 --python .venv/bin/python
!uv pip install onnxruntime-gpu==1.22.0 --python .venv/bin/python
!uv pip install nvidia-cudnn-cu11==8.6.0.163 --python .venv/bin/python
!uv pip install -r {pro_path}/requirements.txt --python .venv/bin/python
!uv pip install pip setuptools --python .venv/bin/python
!uv run paddlex --install hpi-gpu
!uv pip install matplotlib-inline --python .venv/bin/python
!uv pip install wrapt --python .venv/bin/python

Resolved 26 packages in 11.78s
Prepared 26 packages in 1m 58s
Installed 26 packages in 110ms
 + anyio==4.12.1
 + astor==0.8.1
 + certifi==2026.1.4
 + decorator==5.2.1
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.11
 + networkx==3.6.1
 + numpy==2.4.1
 + nvidia-cublas-cu11==11.11.3.6
 + nvidia-cuda-cupti-cu11==11.8.87
 + nvidia-cuda-nvrtc-cu11==11.8.89
 + nvidia-cuda-runtime-cu11==11.8.89
 + nvidia-cudnn-cu11==8.9.6.50
 + nvidia-cufft-cu11==10.9.0.58
 + nvidia-curand-cu11==10.3.0.86
 + nvidia-cusolver-cu11==11.4.1.48
 + nvidia-cusparse-cu11==11.7.5.86
 + nvidia-nccl-cu11==2.19.3
 + nvidia-nvtx-cu11==11.8.86
 + opt-einsum==3.3.0
 + paddlepaddle-gpu==3.0.0
 + pillow==12.1.0
 + protobuf==6.33.4
 + typing-extensions==4.15.0
Resolved 25 packages in 915ms
Prepared 12 packages in 36.19s
Uninstalled 2 packages in 1ms
Installed 12 packages in 258ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 - nvidia-cudnn-cu11==8.9.6.50
 + nvid

In [7]:
# @title 查看帮助
%cd {pro_path}
!{exe_path}/.venv/bin/python ./backend/main.py --help

/kaggle/temp/video-subtitle-remover
Traceback (most recent call last):
  File "/kaggle/temp/video-subtitle-remover/./backend/main.py", line 2, in <module>
    import torch
  File "/kaggle/temp/video-subtitle-remover/.venv/lib/python3.12/site-packages/torch/__init__.py", line 409, in <module>
    from torch._C import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^
ImportError: libcudnn.so.9: cannot open shared object file: No such file or directory


In [8]:
# @title 补丁: GPU字幕检测与无音频兼容
import os

%cd {pro_path}

patch_code = f"""
import sys
import runpy
import subprocess
from paddleocr import TextDetection

# 1. 强制 GPU 模式
_init = TextDetection.__init__
TextDetection.__init__ = lambda self, *a, **k: _init(self, *a, **{{**k, 'device': 'gpu'}})

# 3. 设置参数并运行
sys.argv = [
    './backend/main.py',
    '-i', '{input_path}/input.mp4',
    '-o', '{output_path}/output.mp4',
    '--inpaint-mode', 'sttn-det',
    # '--subtitle-area-coords', '37','153','1516','1885',
    '--subtitle-area-coords', '977','1080','0','1919',
]

runpy.run_path('./backend/main.py', run_name='__main__')
"""

with open('tmp_patch_run.py', 'w') as f:
    f.write(patch_code)

!{exe_path}/.venv/bin/python tmp_patch_run.py

if os.path.exists('tmp_patch_run.py'):
    os.remove('tmp_patch_run.py')

/kaggle/temp/video-subtitle-remover
Traceback (most recent call last):
  File "/kaggle/temp/video-subtitle-remover/tmp_patch_run.py", line 5, in <module>
    from paddleocr import TextDetection
  File "/kaggle/temp/video-subtitle-remover/.venv/lib/python3.12/site-packages/paddleocr/__init__.py", line 15, in <module>
    from paddlex.inference.utils.benchmark import benchmark
  File "/kaggle/temp/video-subtitle-remover/.venv/lib/python3.12/site-packages/paddlex/__init__.py", line 53, in <module>
    from .inference import create_pipeline, create_predictor
  File "/kaggle/temp/video-subtitle-remover/.venv/lib/python3.12/site-packages/paddlex/inference/__init__.py", line 16, in <module>
    from .models import create_predictor
  File "/kaggle/temp/video-subtitle-remover/.venv/lib/python3.12/site-packages/paddlex/inference/models/__init__.py", line 22, in <module>
    from ..utils.official_models import official_models
  File "/kaggle/temp/video-subtitle-remover/.venv/lib/python3.12/site-p

In [9]:
# @title 直接运行
# %cd {pro_path}
# !/content/.venv/bin/python ./backend/main.py -i {input_path}/input-3s.mp4 -o {output_path}/output-3s.mp4 --inpaint-mode sttn-det